# Function 2 - Week 1: what should the first query teach us?

**Taper-waper: start by reducing ignorance, finish by exploiting what we have learned.**

Coverage and hypothesis discrimination answer useful questions, but mixing their scores does not define the risk we are trying to reduce. Here we tie query value to a decision: **how much output might we forgo by choosing the point we currently believe is best?** We then consider what twelve further queries could change.

In [ ]:
from pathlib import Path
import sys

stage = next(p / 'stage-2-bbo' for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'stage-2-bbo/src').is_dir())
sys.path.insert(0, str(stage / 'src'))
from initial_exploration import load_initial_data
from function_2_risk import (analyse_function_2_risk, current_risk_table, query_value_table, final_risk_table, risk_findings, rollout_findings, risk_noise_table, plot_risk_reduction, plot_risk_over_rounds)

data = load_initial_data(stage, function_id=2)
figures = stage.parent / '.runtime/function-2-risk'

## Risk as expected shortfall

We model an underlying mean response f, with noise in each observation. Let G be the finite set of points considered, μ<sub>D</sub>(x) the fitted mean after data D, and x̂ the point with the highest fitted mean.

**x̂ = arg max<sub>x ∈ G</sub> μ<sub>D</sub>(x)**

**R(D) = E[max<sub>x ∈ G</sub> f(x) − f(x̂) | D]**

This is expected simple regret: the output we expect to lose through an imperfect final choice. A risk of 0.10 means **0.10 output units of expected shortfall**, not a 10% chance of failure. It accounts for both the chance and the size of a mistaken choice without inventing a threshold for a “good” result.

The target is the quality of a final recommendation, which may be any point in G. This is not a simulation of the portal's maximum noisy recorded score; restricting the final choice to evaluated inputs would also be a different decision problem.

## Value the answer before knowing it

For a possible query q, consider its possible noisy answers Y<sub>q</sub>, update the model after each answer, and average the remaining risk:

**Expected risk after q = E<sub>Yq | D</sub>[R(D ∪ {(q, Y<sub>q</sub>)})]**

**Value(q) = R(D) − expected risk after q**

Under a fixed probabilistic model this reduction is also

**Value(q) = E<sub>Yq | D</sub>[max<sub>x ∈ G</sub> μ<sub>D+(q,Yq)</sub>(x)] − max<sub>x ∈ G</sub> μ<sub>D</sub>(x).**

This is the **knowledge gradient (KG)**: how much an observation is expected to improve the best decision we can make. A low output can be valuable if it stops us pursuing the wrong region. Disagreement matters when resolving it could change a good decision; an empty region is not automatically valuable just because it is empty. [Knowledge-gradient definition](https://botorch.org/docs/v0.16.0/tutorials/one_shot_kg).

This is a one-query value calculation. It does not, by itself, solve the first-move decision for a thirteen-query budget.

## Three models, three conditional estimates

We retain the three explanations used in the Function 2 exploration: an **x1-only ridge**, **local two-input features** with a shared length scale, and a **flexible two-input surface** with separate length scales. Each is fitted to all ten observations. The main calculation assumes a noise standard deviation of **0.05**, which has not been measured.

The numerical pool contains a 25 × 25 grid, the five earlier candidate points, and the ten supplied input locations. Current risk is estimated from 4,096 joint posterior function draws. Query values integrate over possible answers analytically on that finite pool. No new function observations are obtained.

In [ ]:
risk = analyse_function_2_risk(data, output=figures)

In [ ]:
current_risk_table(risk)

These are separate conditional estimates, not probabilities assigned to the models. The simulation standard error measures the numerical uncertainty in estimating each expectation; it does not cover model error, unknown noise, or peaks between grid points.

In [ ]:
plot_risk_reduction(data, risk, save_to=figures / 'risk-reduction.png')

Colour now measures **expected shortfall reduction in output units**, on the same scale across all panels. There is no coverage-versus-hypothesis weight. Each star is a best tested query under that model; the existing observations retain their rank labels.

In [ ]:
query_value_table(risk)

All three numeric model columns show expected risk reduction from **one** query; larger is better. Values displayed as 0.0000 may be small positive numbers. The last three rows are each model's best tested KG query.

In [ ]:
risk_findings(risk)

## What about the twelve queries still to come?

To check whether the first-move preference survives a longer horizon, simulate **128 possible functions per model**. Compare eight first queries. After each first answer, take twelve further queries with the same declared policy:

**Next query = arg max [updated mean + κ × updated latent standard deviation]**

κ decreases linearly from **2 to 0** over those twelve queries. Each simulated observation updates the Gaussian posterior; noise is included, repeated queries are allowed, and model parameters stay fixed. The final recommendation maximises the updated mean over the whole pool.

Within each model, all first-query options share the same simulated functions and noise draws, making comparisons paired. This is a bounded test of one taper policy, not an optimal thirteen-step planner. Values below are **simulated expected shortfall ± one simulation standard error**; lower is better.

In [ ]:
plot_risk_over_rounds(risk, save_to=figures / 'risk-over-rounds.png')

In [ ]:
final_risk_table(risk)

In [ ]:
rollout_findings(risk)

## How sensitive is this to noise and grid resolution?

The table repeats the one-query calculation with smaller and larger assumed noise. These checks do not repeat the thirteen-query simulation. The upper-right remains attractive to the local model, while the other models can favour different locations.

In [ ]:
risk_noise_table(risk)

The finer 41 × 41 grid check preserves the broad preferred regions but changes individual coordinates and estimated gains. All quoted risks are relative to a finite pool, and the conditional calculations hold fitted parameters fixed. A more complete analysis could also allow parameters and model preferences to change after new observations.

Most importantly, **none of these models can give a model-free bound on what an unseen region contains**. Ten observations cannot rule out an arbitrarily narrow, high peak. A risk estimate becomes numerical because we have specified possible behaviour, not because the data have removed that uncertainty.

## What this changes about the taper

The first query should reduce uncertainty that matters to a later decision. Coverage, disagreement and promising outputs can each help do that, but they are proxies rather than interchangeable definitions of risk.

On this conditional one-query criterion, the upper-right region deserves priority over the earlier 50:50 coverage blend. The longer-horizon check leaves room for several defensible starts. The useful commitment is to a clear loss, explicit assumptions and decisions that update with evidence.

**Status: risk comparison complete; no portal submission has been made.**